## Exceedance probability for total precipitation compared to 2026.  

In [30]:
import numpy as np
import pandas as pd
import matplotlib as plt
import scipy as pi


df = pd.read_csv('Data/mores_data.csv')
df.head()


,Date,SWE (in),Snow Depth (in),Avg Air Temp (deg F),Avg Relative Humidity (%),Accumulated Precip (in),Annual Precip (in),Annual Precip Exceedance (%)
0,1980-11-03,NaN,NaN,NaN,NaN,1.3,1.3,NaN
1,1980-11-04,NaN,NaN,NaN,NaN,NaN,1.3,NaN
2,1980-11-05,NaN,NaN,NaN,NaN,NaN,1.3,NaN
3,1980-11-06,NaN,NaN,NaN,NaN,NaN,1.3,NaN
4,1980-11-07,NaN,NaN,NaN,NaN,NaN,1.3,NaN


In [20]:
df = pd.read_csv("Data/mores_data.csv")
df["Date"] = pd.to_datetime(df["Date"])

year = df["Date"].dt.year  # Calendar year
annual_totals = df.groupby(year)["Accumulated Precip (in)"].sum(min_count=1)
df["Annual Precip (in)"] = year.map(annual_totals)

df.to_csv("Data/mores_data.csv", index=False)

- Use a Weibull plotting position to estimate the exceedance probability for each annual precipitation total:Rank the largest annual total first (rank 1). Since your annual total is repeated on every daily row, calculate ranks on one row per year, not on the daily data.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

# Water year ends in the named year: Oct 1980–Sep 1981 is 1981
df["Year"] = df["Date"].dt.year + (df["Date"].dt.month >= 10)

annual = (
    df.groupby("Year", as_index=False)["Accumulated Precip (in)"]
      .sum(min_count=1)
      .rename(columns={"Accumulated Precip (in)": "Annual Precip (in)"})
      .dropna(subset=["Annual Precip (in)"])
)

# Exclude 1980 and 1981 before ranking
annual = annual.loc[~annual["Year"].isin([1980, 1981])].copy()

n = len(annual)
if n == 0:
    raise ValueError("No annual precipitation values to rank.")

annual["Rank"] = annual["Annual Precip (in)"].rank(
    ascending=False, method="average"
)
annual["Exceedance Probability (%)"] = annual["Rank"] / (n + 1) * 100

exceedance_precip_annual = annual.sort_values(
    "Annual Precip (in)", ascending=False
)
display(exceedance_precip_annual)



,Year,Annual Precip (in),Rank,Exceedance Probability (%)
16,1997,16673.3,1.0,2.127660
15,1996,14633.9,2.0,4.255319
36,2017,14612.8,3.0,6.382979
25,2006,13426.5,4.0,8.510638
2,1983,13409.8,5.0,10.638298
5,1986,13357.6,6.0,12.765957
30,2011,13150.5,7.0,14.893617
3,1984,12559.7,8.0,17.021277
14,1995,12495.6,9.0,19.148936
1,1982,12267.1,10.0,21.276596


In [14]:
probability_by_year = annual.set_index("Year")["Exceedance Probability (%)"]
df["Annual Precip Exceedance (%)"] = (
    df["Date"].dt.year.map(probability_by_year)
)

df.to_csv("Data/mores_data.csv", index=False)

### Exceedance probability for stream flow volume compared to 2026. 

In [7]:
df2 = pd.read_csv('Data/Twinsprings_alltime.csv')
df2.head()

/var/folders/9n/npt_hczs5j115291dbmz7jnh0000gn/T/ipykernel_60544/1084890759.py:1: DtypeWarning: Columns (3,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('Data/Twinsprings_alltime.csv')


,id,approval_status,last_modified,method_category,monitoring_location_id,parameter_code,qualifier,statistic_id,time,time_series_id,unit_of_measure,value,x,y
0,65073363-2d0f-483a-8783-3e3c2205e93d,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T06:00:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,604.0,-115.725278,43.668056
1,12a871d8-d5d0-47db-9bb0-95ebaa85d9d9,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T06:15:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,648.0,-115.725278,43.668056
2,347b8fca-7d81-45bf-9213-1b16033b2e6b,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T06:30:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,604.0,-115.725278,43.668056
3,25aaa074-ad6e-4780-a34e-0dcb4d1488e0,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T06:45:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,598.0,-115.725278,43.668056
4,a11ab19e-dff6-4883-ac96-291b6f08926a,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T07:00:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,598.0,-115.725278,43.668056


- Annual Streamflow Depth (mm), Annual Streamflow Volume (m3)
The values use discharge records in ft^3/s, integrate consecutive 15-minute readings, and convert the resulting annual volume to basin-average depth using the 2,154.9 km² watershed area. Each year’s totals are repeated on the rows for that year. The watershed area is found by converting the watershed area 832 square miles from USGS gauge website to km2. 

In [ ]:
path = 'Data/Twinsprings_alltime.csv'
watershed_area_km2 = 2154.9

# Read raw discharge measurements only (not the ft gage-height records)
flow = pd.read_csv(
    path,
    usecols=["time", "unit_of_measure", "value"]
)
flow = flow.loc[flow["unit_of_measure"] == "ft^3/s"].copy()

# Convert UTC timestamps to local Twin Springs time
flow["time"] = pd.to_datetime(flow["time"], utc=True).dt.tz_convert(
    "America/Boise"
)
flow["value"] = pd.to_numeric(flow["value"], errors="coerce")
flow = (
    flow.dropna(subset=["time", "value"])
        .sort_values("time")
        .reset_index(drop=True)
)

# Integrate each consecutive 15-minute interval using mean discharge
interval = flow["time"].shift(-1) - flow["time"]
next_flow = flow["value"].shift(-1)
valid = interval.eq(pd.Timedelta(minutes=15)) & next_flow.notna()

midpoint = flow["time"] + interval / 2
interval_volume_m3 = (
    (flow["value"] + next_flow) / 2
    * interval.dt.total_seconds()
    * 0.028316846592  # cubic feet to cubic meters
)

intervals = pd.DataFrame({
    "Water Year": midpoint.dt.year
                  + (midpoint.dt.month >= 10).astype(int),
    "Volume (m3)": interval_volume_m3,
}).loc[valid]

annual_flow = (
    intervals.groupby("Water Year", as_index=False)["Volume (m3)"]
             .sum()
)

# Convert annual volume to basin-average depth
annual_flow["Depth (mm)"] = (
    annual_flow["Volume (m3)"]
    / (watershed_area_km2 * 1_000_000)
    * 1_000
)

# Omit the incomplete first 
annual_flow = annual_flow.loc[
    ~annual_flow["Water Year"].isin([1987])
].copy()

n = len(annual_flow)
if n == 0:
    raise ValueError("No complete water years are available to rank.")

annual_flow["Depth Exceedance Probability (%)"] = (
    annual_flow["Depth (mm)"].rank(ascending=False, method="average")
    / (n + 1) * 100
)
annual_flow["Volume Exceedance Probability (%)"] = (
    annual_flow["Volume (m3)"].rank(ascending=False, method="average")
    / (n + 1) * 100
)

annual_flow = annual_flow.sort_values("Volume (m3)", ascending=False)
display(annual_flow)

/var/folders/9n/npt_hczs5j115291dbmz7jnh0000gn/T/ipykernel_60544/577856938.py:25: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  valid = interval.eq(pd.Timedelta(minutes=15)) & next_flow.notna()


,Water Year,Volume (m3),Depth (mm),Depth Exceedance Probability (%),Volume Exceedance Probability (%)
30,2017.0,1.921915e+09,891.881277,2.5,2.5
10,1997.0,1.810946e+09,840.385231,5.0,5.0
9,1996.0,1.482007e+09,687.738223,7.5,7.5
24,2011.0,1.435013e+09,665.930291,10.0,10.0
19,2006.0,1.330251e+09,617.314308,12.5,12.5
25,2012.0,1.291907e+09,599.520635,15.0,15.0
32,2019.0,1.176816e+09,546.111458,17.5,17.5
36,2023.0,1.137117e+09,527.688928,20.0,20.0
11,1998.0,1.125771e+09,522.423834,22.5,22.5
31,2018.0,1.112645e+09,516.332669,25.0,25.0
